In [2]:
from PIL import Image
import imagehash
import cv2

In [3]:
def compute_frame_phash(gray_frame):
    pil_image = Image.fromarray(gray_frame)
    
    phash = imagehash.phash(pil_image, hash_size=16, highfreq_factor=16)
    return phash

In [4]:
def preprocess_frame(frame, blur_kernel_size=(5, 5)):
    """Preprocess the frame by converting to grayscale and applying Gaussian blur."""
    try:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    except cv2.error as e:
        print(f"Error converting frame to grayscale: {e}")
        print(f"Frame shape: {frame.shape}, Frame type: {type(frame)}")
        return None
    
    frame = cv2.GaussianBlur(frame, blur_kernel_size, 0)

    return frame

In [5]:
def get_frame_hashes(video_path):
    cap = cv2.VideoCapture(video_path)
    phash=[]

    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f'FPS : {fps:0.2f}')

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame = preprocess_frame(frame)
        phash.append(compute_frame_phash(frame))

    cap.release()
    return phash, fps

In [6]:
OLD_VIDEO='main.mp4'
NEW_VIDEO='deleted_15_frames.mp4'

In [7]:
phash1, fps1= get_frame_hashes(OLD_VIDEO)

FPS : 30.00


In [8]:
phash2, fps2 = get_frame_hashes(NEW_VIDEO)

FPS : 30.00


In [9]:
len(phash1), len(phash2)

(685, 670)

In [10]:
video_id=1
init=True

In [11]:
def is_significantly_different(hash1, hash2, threshold=5):
    return hash1 - hash2 > threshold

In [12]:
i,j=0,0
deleted=[]
added=[]
changedFrames=[]
while i<len(phash1) and j<len(phash2):
    currphash1=phash1[i]
    currphash2=phash2[j]
    if is_significantly_different(currphash1, currphash2):
        flag=True
        if (len(phash1)!=len(phash2)):
            for k in range(1,60): # check changes for 2 seconds as of now
                if i + k >= len(phash1): # out of bounds
                    break
                newphash1=phash1[i+k]
                if newphash1==currphash2:
                    # if k+1<15: # doubt-----------------------------------------------------------
                    #     break
                    print("-----------Deleted Frames-----------")
                    print(i, i+k, i/30,"sec", "-", (i+k)/30,"sec")
                    deleted.append((i,i+k-1))
                    i=i+k
                    # print(i/30)
                    print("----------------------")
                    flag=False
                    break
            
            if flag:
                for k in range(1,60):
                    if j + k >= len(phash2):
                        break
                    newphash2=phash2[j+k]
                    if currphash1==newphash2:
                        # if k+1<15: # doubt-----------------------------------------------------------
                        #     break
                        print("-----------Added Frames-----------")
                        print(j, j+k, j/30,"sec", "-", (j+k)/30,"sec")
                        added.append((j,j+k-1))
                        j=j+k
                        # print(j/30)
                        print("----------------------")
                        flag=False
                        break
                
        if flag:
            changedFrames.append(j)
            # print("J=",j)
            
    i+=1
    j+=1
print("Deleted Frames",deleted)
print("Added Frames", added)
print("Changes Frames", changedFrames)

-----------Deleted Frames-----------
150 165 5.0 sec - 5.5 sec
----------------------
Deleted Frames [(150, 164)]
Added Frames []
Changes Frames []


In [13]:
import os

def get_previous_length_of_streams(frames1_length: int):
    file_path = f"{video_id}/previousContext/previousLengthOfStreams.txt"
    
    if os.path.exists(file_path) or init==False:
        # Read from file if it exists
        with open(file_path, "r") as file:
            previous_length_of_streams = [int(line.strip()) for line in file.readlines()]
    else:
        # Compute and store if file doesn't exist
        previous_length_of_streams = []
        i = frames1_length
        while i >= 0 and i // 300 != 0:
            previous_length_of_streams.append(300)
            i -= 300

        if i % 300 != 0:
            previous_length_of_streams.append(i % 300)

        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        with open(file_path, "w") as file:
            for length in previous_length_of_streams:
                file.write(f"{length}\n")

    return previous_length_of_streams

In [14]:
def update_previous_length_of_streams(new_length_of_streams):
    file_path = f"{video_id}/previousContext/previousLengthOfStreams.txt"
    
    try:
        # Create directory if it doesn't exist
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        
        # Write new segment lengths to file, overwriting existing content
        with open(file_path, "w") as file:
            for length in new_length_of_streams:
                file.write(f"{length}\n")
        
        print(f"Successfully updated {file_path} with lengths: {new_length_of_streams}")
        return True
    
    except Exception as e:
        print(f"Error updating {file_path}: {e}")
        return False

In [15]:
previous_length_of_streams= get_previous_length_of_streams(len(phash1))
previous_length_of_streams

[285, 300, 85]

In [16]:
new_length_of_streams=previous_length_of_streams[:] ## first copy it into new length and then calculated the revised length of streams

In [17]:
changedSegments=set()

In [18]:
# For deleted Frames
for deletedFrame in deleted:
    start=deletedFrame[0]
    end=deletedFrame[1]

    index=start//300
    changedSegments.add(index)
    print(index, previous_length_of_streams[index])
    new_length_of_streams[index]-=end-start+1

print(new_length_of_streams)
print(changedSegments)

0 285
[270, 300, 85]
{0}


In [19]:
# For added frames
for addedFrame in added:
    start=addedFrame[0]
    end=addedFrame[1]

    index=start//300
    changedSegments.add(index)
    print(index, previous_length_of_streams[index])
    new_length_of_streams[index]+=end-start+1

print(new_length_of_streams)
print(changedSegments)

[270, 300, 85]
{0}


In [20]:
for frame in changedFrames:
    index=frame//300
    changedSegments.add(index)
    print(index, frame)
print(new_length_of_streams)
print(changedSegments)

[270, 300, 85]
{0}


In [21]:
# Put keyframes at new_length_of_streams
# ffmpeg -i deleted_15_frames.mp4 -c:v libx264 -force_key_frames "expr:eq(n,285)+eq(n,585)" -sc_threshold 0 -c:a aac -r 30 deleted_cleaned_input.mp4
import subprocess

output_file = "temp_new_video.mp4"
def put_keyframes_at_new_length_of_streams(new_length_of_streams):
    keyframe_frames = [0]  # Start at frame 0
    cumulative_frames = 0
    for frames in new_length_of_streams:
        cumulative_frames += frames
        keyframe_frames.append(cumulative_frames)
    keyframe_frames.pop()  
    
    expressions = [f"eq(n,{frame})" for frame in keyframe_frames[1:]]  # Skip 0
    keyframe_expr = "+".join(expressions) if expressions else "0"  # e.g., "eq(n,284)+eq(n,584)"
    
    # Build FFmpeg command
    # ffmpeg_cmd = [
    #     "ffmpeg",
    #     "-i", NEW_VIDEO,
    #     "-c:v", "libx264",
    #     "-force_key_frames", f"expr:{keyframe_expr}",
    #     "-sc_threshold", "0",
    #     "-vf", "setpts=PTS-STARTPTS",
    #     "-c:a", "aac",
    #     "-af", "asetpts=PTS-STARTPTS",
    #     "-r", "30",
    #     "-vsync", "1",
    #     "-y",  # Overwrite output
    #     output_file
    # ]
    ffmpeg_cmd = [
        "ffmpeg",
        "-i", NEW_VIDEO,
        "-c:v", "libx264",
        "-force_key_frames", f"expr:{keyframe_expr}",
        "-sc_threshold", "0",
        "-g", "1000",  # Large GOP to prevent extra keyframes
        "-keyint_min", "1000",
        "-vf", "setpts=PTS-STARTPTS",
        "-c:a", "aac",
        "-af", "asetpts=PTS-STARTPTS",
        "-r", "30",
        "-vsync", "1",
        "-y",
        output_file
    ]
    
    try:
        # Run FFmpeg command
        subprocess.run(ffmpeg_cmd, check=True, capture_output=True, text=True)
        print(f"Successfully created {output_file} with keyframes at frames: {keyframe_frames}")
        return True
    except subprocess.CalledProcessError as e:
        print(f"Error running FFmpeg: {e.stderr}")
        return False

In [22]:
put_keyframes_at_new_length_of_streams(new_length_of_streams)

Successfully created temp_new_video.mp4 with keyframes at frames: [0, 270, 570]


True

In [23]:
# ffprobe -loglevel error -select_streams v:0 -show_entries packet=pts_time,flags -of csv=print_section=0 input.mp4

def get_keyframes(file_path):
    cmd = [
        "ffprobe",
        "-loglevel", "error",
        "-select_streams", "v:0",
        "-show_entries", "packet=pts_time,flags",
        "-of", "csv=print_section=0",
        file_path
    ]

    try:
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)
        print(result)
        keyframes = []
        for line in result.stdout.splitlines():
            # print(line)
            pts_time, flags = line.split(",")
            if flags.__contains__("K"):
                keyframes.append(float(pts_time))
        return keyframes
    except subprocess.CalledProcessError as e:
        print(f"Error running ffprobe: {e.stderr}")
        return []

In [24]:
keyframes=get_keyframes(output_file)
print("Keyframes in the new video:")
for keyframe in keyframes:
    print(keyframe)

CompletedProcess(args=['ffprobe', '-loglevel', 'error', '-select_streams', 'v:0', '-show_entries', 'packet=pts_time,flags', '-of', 'csv=print_section=0', 'temp_new_video.mp4'], returncode=0, stdout='0.000000,K__\n0.100000,___\n0.033333,___\n0.066667,___\n0.233333,___\n0.166667,___\n0.133333,___\n0.200000,___\n0.366667,___\n0.300000,___\n0.266667,___\n0.333333,___\n0.500000,___\n0.433333,___\n0.400000,___\n0.466667,___\n0.633333,___\n0.566667,___\n0.533333,___\n0.600000,___\n0.766667,___\n0.700000,___\n0.666667,___\n0.733333,___\n0.900000,___\n0.833333,___\n0.800000,___\n0.866667,___\n1.033333,___\n0.966667,___\n0.933333,___\n1.000000,___\n1.166667,___\n1.100000,___\n1.066667,___\n1.133333,___\n1.300000,___\n1.233333,___\n1.200000,___\n1.266667,___\n1.433333,___\n1.366667,___\n1.333333,___\n1.400000,___\n1.566667,___\n1.500000,___\n1.466667,___\n1.533333,___\n1.700000,___\n1.633333,___\n1.600000,___\n1.666667,___\n1.733333,___\n1.866667,___\n1.800000,___\n1.766667,___\n1.833333,___\n2.0

In [25]:
# ffmpeg -allowed_extensions ALL -i playlist.m3u8 -r 30 -c:v copy -c:a copy merged.mp4
merged_output_file = "merged_output.mp4"
def merge_segments():
    ffmpeg_cmd = [
        "ffmpeg",
        "-allowed_extensions", "ALL",
        "-i", "playlist.m3u8",
        "-r", "30",
        "-c:v", "copy",
        "-c:a", "copy",
        "-y",  # Overwrite output
        merged_output_file
    ]
    try:
        # Run FFmpeg command
        result = subprocess.run(ffmpeg_cmd, check=True, capture_output=True, text=True)
        print(f"Successfully merged segments into {output_file}")
        return True
    except subprocess.CalledProcessError as e:
        print(f"Error merging segments: {e.stderr}")
        return False

In [26]:
merge_segments()

Successfully merged segments into temp_new_video.mp4


True

In [27]:
update_previous_length_of_streams(new_length_of_streams)

Successfully updated 1/previousContext/previousLengthOfStreams.txt with lengths: [270, 300, 85]


True